# 07 — FINAL test (chỉ chạy khi đủ cổng)

**Mở test đúng MỘT lần; chỉ khi N4 đổi `FINAL=True` và Restart & Run All.**
Mọi bảng ngoài mục 7 của protocol phải ghi nhãn **thăm dò (post-hoc)**.

Xem `reports/final_protocol.md` mục 8.
Notebook này dừng ở cổng GATE / PRE-FLIGHT nếu thiếu điều kiện; chưa đọc `test.csv`.

In [ ]:
import os, sys
from pathlib import Path
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, os.getcwd())

FINAL = False                      # N4 doi thanh True khi chay chinh thuc — ban KHONG doi
PROTOCOL_TAG  = "protocol-v1"
PROTOCOL_MD   = "reports/final_protocol.md"
RESULTS       = "reports/results"
FROZEN        = f"{RESULTS}/frozen_thresholds.csv"
MODELS        = ["dummy", "logreg", "tree", "lightgbm_smotenc", "Tung_mlp_v2"]
LOCKED_HASHES = {  # tu protocol muc 2 — tinh sha256 de doi chieu, KHONG doc noi dung test
    "data/processed/train.csv": "4e75d796b7e993bb350d202d428f9e3ad9d746daa3e42704deb0cbb21482329e",
    "data/processed/val.csv":   "1fcbee5f639ac363bbbf155235d2cad45436d5e3dd7a73b1c1e422289c3485c1",
    "data/processed/test.csv":  "2cac5114304eb69a10b47f34881ded19dfbd5bbcb3d0a9ed107b438c100ae200",
    "models/Tung_mlp_v2.joblib":"5090cbc6273c034ad4446a69b9f8e728217a6a956084de504ca248553759b970",
}

In [2]:
import subprocess
r = subprocess.run(["git", "rev-parse", "--verify", f"refs/tags/{PROTOCOL_TAG}"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("DỪNG: chưa có tag protocol-v1")
    raise SystemExit(1)
PROTOCOL_HASH = r.stdout.strip()
print(f"GATE 1 OK — tag protocol-v1 -> {PROTOCOL_HASH}")

DỪNG: chưa có tag protocol-v1


SystemExit: 1

/home/user/venv-smartstay/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
from pathlib import Path
import pandas as pd

if not Path(PROTOCOL_MD).exists():
    print("DỪNG: thiếu reports/final_protocol.md")
    raise SystemExit(1)
if not Path(FROZEN).exists():
    print("DỪNG: thiếu reports/results/frozen_thresholds.csv")
    raise SystemExit(1)

df = pd.read_csv(FROZEN)
required_models = [m for m in MODELS if m != "dummy"]
RATIOS = {0.5, 1, 2, 3, 5, 10}
missing = []
for m in required_models:
    sub = df[df["model"] == m]
    have = set(sub["r"].astype(float))
    lacking = sorted(RATIOS - have)
    if lacking:
        missing.append((m, lacking))

if missing:
    print("DỪNG: frozen_thresholds.csv thiếu dòng:")
    for m, t in missing:
        print(f"  - {m}: r còn thiếu {t}")
    raise SystemExit(1)
print(f"GATE 2 OK — frozen_thresholds.csv đủ 6 dòng r x {len(required_models)} model")

In [ ]:
import subprocess
r = subprocess.run(["git", "diff", "--quiet", "protocol-v1", "--", "src/", "reports/final_protocol.md"])
if r.returncode != 0:
    r2 = subprocess.run(["git", "diff", "--stat", "protocol-v1", "--", "src/", "reports/final_protocol.md"],
                        capture_output=True, text=True)
    print("DỪNG: src/ hoặc reports/final_protocol.md khác với tag protocol-v1:")
    print(r2.stdout)
    raise SystemExit(1)
print("GATE 3 OK — src/ và reports/final_protocol.md không đổi kể từ protocol-v1")

In [ ]:
import hashlib

for p, expected in LOCKED_HASHES.items():
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    actual = h.hexdigest()
    ok = actual == expected
    print(f"{'OK  ' if ok else 'LECH'} {p}")
    print(f"      ky vong: {expected}")
    print(f"      thuc te: {actual}")
    if not ok:
        print("DỪNG: hash lệch so với LOCKED_HASHES")
        raise SystemExit(1)
print("GATE 4 OK — toàn bộ hash khớp")

In [ ]:
import numpy as np
from pathlib import Path

missing = []
for m in MODELS:
    if m == "dummy":
        continue
    mf, pf = f"models/{m}.joblib", f"models/proba_val_{m}.npy"
    if not Path(mf).exists():
        missing.append(mf)
    if not Path(pf).exists():
        missing.append(pf)
    else:
        try:
            n = len(np.load(pf))
            if n != 14400:
                missing.append(f"{pf} (len={n} != 14400)")
        except Exception as e:
            missing.append(f"{pf} (loi doc: {e})")

if missing:
    print("DỪNG: thiếu model/proba:")
    for x in missing:
        print(f"  - {x}")
    raise SystemExit(1)
print("GATE 5 OK — models/proba đủ: " + ", ".join(m for m in MODELS if m != "dummy"))

In [ ]:
from pathlib import Path

if Path("reports/results/test_final.csv").exists():
    print("DỪNG: đã có test_final.csv — không chạy lại")
    raise SystemExit(1)
print("GATE 6 OK — chưa có reports/results/test_final.csv")

In [ ]:
if FINAL is False:
    print("PRE-FLIGHT PASSED. Chỉ N4 mới được đổi FINAL=True + Restart & Run All.")
    raise SystemExit

## PHẦN 2 — cells FINAL=True (viết sẵn, KHÔNG chạy ở Phần này)

Các cell dưới đây CHỈ được chạy khi N4 đổi cờ FINAL ở Cell 1 thành True và Restart & Run All. Tại thời điểm Phần 3–4, notebook dừng ở Cell 8 (PRE-FLIGHT).

Mọi bảng ngoài mục 7 protocol phải ghi nhãn **thăm dò (post-hoc)**. Xem `reports/final_protocol.md` mục 7–8.

In [ ]:
from src.data import load_split
test = load_split("test", allow_test=True)
X_test = test.drop(columns=["is_canceled"])
y_test = test["is_canceled"]
print(f"n_test = {len(y_test)} | cancel_rate = {y_test.mean():.4f}")   # lan dau nhin nhan test

In [ ]:
import numpy as np
import joblib
from src.data import load_xy

model = joblib.load("models/Tung_mlp_v2.joblib")
proba_val = np.load("models/proba_val_Tung_mlp_v2.npy")
Xv, _ = load_xy("val")
p_val = model.predict_proba(Xv)[:, 1]
max_abs_diff = float(np.abs(p_val - proba_val).max())
print(f"max_abs_diff = {max_abs_diff:.3e}")
if max_abs_diff > 1e-06:
    print("DỪNG: proba lệch > 1e-06 — báo N4")
    raise SystemExit(1)

In [ ]:
import numpy as np
import joblib
from src.data import load_xy

y_train_mean = float(load_xy("train")[1].mean())
proba_test = {}
print(f"y_train_mean = {y_train_mean:.6f} | n_test = {len(y_test)}")
for m in MODELS:
    if m == "dummy":
        proba = np.full(len(y_test), y_train_mean)
    else:
        proba = joblib.load(f"models/{m}.joblib").predict_proba(X_test)[:, 1]
    proba_test[m] = proba
    np.save(f"models/proba_test_{m}.npy", proba.astype("float32"))
    print(m, proba.shape, proba.dtype)

In [ ]:
import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss
from src.evaluate import evaluate

_frozen = pd.read_csv(FROZEN)
_rows = []
for m in MODELS:
    proba = proba_test[m]
    if m == "dummy":
        t = 0.99
    else:
        t = float(_frozen[(_frozen.model == m) & (_frozen.r == 2.0)]["threshold"].iloc[0])
    ev = evaluate(y_test, proba, threshold=t)
    ev.update({"model": m,
               "brier": brier_score_loss(y_test, proba),
               "log_loss": log_loss(y_test, proba),
               "mean_proba": float(np.mean(proba))})
    _rows.append(ev)

test_final = pd.DataFrame(_rows)[["model", "n", "positive_rate", "threshold", "accuracy",
                                   "precision", "recall", "f1", "roc_auc", "pr_auc",
                                   "tn", "fp", "fn", "tp", "brier", "log_loss", "mean_proba"]]
test_final.to_csv(f"{RESULTS}/test_final.csv", index=False)
print(test_final.to_string(index=False))

In [ ]:
from src.cost import cost_at_threshold, baseline_costs

never = baseline_costs(y_test, 2.0, 1.0)["never_overbook"]
print(f"baseline never_overbook cost = {never}")

tf = pd.read_csv(f"{RESULTS}/test_final.csv")
cost_locked, cost_never_col, saving_col = [], [], []
for m in MODELS:
    proba = proba_test[m]
    t = float(tf[tf.model == m]["threshold"].iloc[0])
    c = cost_at_threshold(y_test, proba, t, c_walk=2.0, c_empty=1.0)["cost"]
    cost_locked.append(c)
    cost_never_col.append(never)
    saving_col.append(float((never - c) / never * 100.0) if never else float("nan"))
    if m == "Tung_mlp_v2":
        print(f"MLP cost @0.50 = {cost_at_threshold(y_test, proba, 0.5, 2.0, 1.0)['cost']}")
        print(f"MLP cost @2/3  = {cost_at_threshold(y_test, proba, 2/3, 2.0, 1.0)['cost']}")

tf["cost_r2_locked"] = cost_locked
tf["cost_never"] = cost_never_col
tf["saving_vs_never_pct"] = saving_col
tf.to_csv(f"{RESULTS}/test_final.csv", index=False)
print(tf[["model", "cost_r2_locked", "cost_never", "saving_vs_never_pct"]].to_string(index=False))

In [ ]:
from src.cost import cost_at_threshold

_frozen2 = pd.read_csv(FROZEN)
sub = _frozen2[_frozen2.model == "Tung_mlp_v2"]
rows = []
for _, rr in sub.iterrows():
    t = float(rr["threshold"])
    c = cost_at_threshold(y_test, proba_test["Tung_mlp_v2"], t, c_walk=2.0, c_empty=1.0)
    rows.append({"model": "Tung_mlp_v2", "r": float(rr["r"]), "threshold": t,
                 "cost": c["cost"], "cost_per_booking": c["cost_per_booking"]})
sens = pd.DataFrame(rows)
sens.to_csv(f"{RESULTS}/test_final_sensitivity.csv", index=False)
print(sens.to_string(index=False))

Bảng độ nhạy theo r (mục 7.3); **không dùng để chọn ngưỡng** — ngưỡng đã khóa từ frozen_thresholds.csv.

In [ ]:
import numpy as np

t_mlp = float(_frozen[(_frozen.model == "Tung_mlp_v2") & (_frozen.r == 2.0)]["threshold"].iloc[0])
proba = proba_test["Tung_mlp_v2"]

for col, fname in [("hotel", "test_final_by_hotel.csv"), ("arrival_date_month", "test_final_by_month.csv")]:
    rows = []
    for g in sorted(X_test[col].unique()):
        mask = (X_test[col] == g).to_numpy()
        yg, pg = y_test.to_numpy()[mask], proba[mask]
        ev = evaluate(yg, pg, threshold=t_mlp)
        c = cost_at_threshold(yg, pg, t_mlp, 2.0, 1.0)["cost"]
        rows.append({"group": g, "model": "Tung_mlp_v2", **ev, "cost_r2_locked": c})
    out = pd.DataFrame(rows)
    out.to_csv(f"{RESULTS}/{fname}", index=False)
    print(fname)
    print(out.to_string(index=False))
print("Luu y: khong suy dien nhan qua; thang la dac trung, khong phai nguyen nhan.")

In [ ]:
from pathlib import Path
from src.final_eval import paired_bootstrap

if Path("models/proba_test_lightgbm_smotenc.npy").exists():
    ref = "lightgbm_smotenc"
    note = "so voi lightgbm_smotenc"
else:
    ref = "logreg"
    note = "lightgbm_smotenc khong co trong MODELS lan chay (can quyet dinh chinh thuc cua nhom) — doi chieu tam voi logreg"

pb = paired_bootstrap(y_test.to_numpy(), proba_test["Tung_mlp_v2"], proba_test[ref])
row = {"reference_model": ref, "note": note,
       "n_boot_used": pb["n_boot_used"], "n_boot_skipped": pb["n_boot_skipped"],
       "diff_pr_auc": pb["diff_pr_auc"],
       "ci_pr_auc_lo": pb["ci_pr_auc"][0], "ci_pr_auc_hi": pb["ci_pr_auc"][1],
       "diff_roc_auc": pb["diff_roc_auc"],
       "ci_roc_auc_lo": pb["ci_roc_auc"][0], "ci_roc_auc_hi": pb["ci_roc_auc"][1],
       "frac_boot_a_better_pr": pb["frac_boot_a_better_pr"],
       "frac_boot_a_better_roc": pb["frac_boot_a_better_roc"]}
boot = pd.DataFrame([row])
boot.to_csv(f"{RESULTS}/test_final_bootstrap.csv", index=False)
print(boot.to_string(index=False))

In [ ]:
print(f"y_test.mean() = {y_test.mean():.6f}")
for m in MODELS:
    print(f"{m:15s} mean_proba_test = {proba_test[m].mean():.6f}")

In [ ]:
import hashlib, sys, subprocess
import numpy as np
import sklearn
import pandas as pd
import joblib
from datetime import datetime

now = datetime.now().strftime("%Y-%m-%d %H:%M")
lines = [f"## Lan chay FINAL=True — {now}",
         f"datetime: {now}",
         f"git HEAD: {subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()}",
         f"git describe --tags: {subprocess.run(['git', 'describe', '--tags'], capture_output=True, text=True).stdout.strip()}"]
for p, expected in LOCKED_HASHES.items():
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    actual = h.hexdigest()
    lines.append(f"sha256 {p}: {actual} ({'MATCH' if actual == expected else 'LECH'})")
lines += [f"python: {sys.version.split()[0]}",
          f"numpy {np.__version__} | scikit-learn {sklearn.__version__} | pandas {pd.__version__} | joblib {joblib.__version__}",
          "Nguoi chay: N4",
          "Nguoi chung kien: N1"]
with open(f"{RESULTS}/test_run_log.md", "a", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n\n")
print("\n".join(lines))

Báo cáo đúng như đã chạy, **dù kết quả xấu**; không đổi kết luận chính. Phân tích thêm chỉ được ghi là **thăm dò (post-hoc)**.